# Interactions

An interaction lets the slope of one predictor depend on the value of another. The product $x z$ is just another column of the design matrix. The model is still linear in its coefficients.


## A two-by-two table

Four designed corners, with $x$ and $z$ each equal to $0$ or $1$:

$$
\begin{array}{c|cc}
z \backslash x & 0 & 1 \\
\hline
0 & 1 & 3 \\
1 & 2 & 8
\end{array}
$$

Fit

$$
y = b_0 + b_1 x + b_2 z + b_3 xz.
$$

Four coefficients and four points determine the fit by substitution, starting where the predictors vanish.

$$
\begin{aligned}
(x, z) = (0, 0) &\Longrightarrow b_0 = 1, \\
(1, 0) &\Longrightarrow b_0 + b_1 = 3 \Longrightarrow b_1 = 2, \\
(0, 1) &\Longrightarrow b_0 + b_2 = 2 \Longrightarrow b_2 = 1, \\
(1, 1) &\Longrightarrow b_0 + b_1 + b_2 + b_3 = 8 \Longrightarrow b_3 = 4.
\end{aligned}
$$

The slope of $x$ is $b_1 + b_3 z$. It equals $2$ when $z = 0$ and $2 + 4 = 6$ when $z = 1$. The interaction is the amount by which the slope of $x$ changes when $z$ increases by one.


In [1]:
# Substitution at the four corners recovers the interaction model.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
y = np.array([1.0, 3, 2, 8])
X = np.column_stack([np.ones(4), x, z, x * z])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta", beta)
print("slope of x when z=0", beta[1])
print("slope of x when z=1", beta[1] + beta[3])
print("RSS", np.sum((y - X @ beta) ** 2))
assert np.allclose(beta, [1, 2, 1, 4])
assert abs((beta[1] + beta[3]) - 6) < 1e-12
assert np.sum((y - X @ beta) ** 2) < 1e-12


beta [1. 2. 1. 4.]
slope of x when z=0 2.0
slope of x when z=1 6.0
RSS 0.0


## The same table without the product

Drop $xz$ and fit $b_0 + b_1 x + b_2 z$. The normal equations are

$$
X^{\mathsf T}X = \begin{bmatrix} 4 & 2 & 2 \\ 2 & 2 & 1 \\ 2 & 1 & 2 \end{bmatrix}, \qquad
X^{\mathsf T}y = \begin{bmatrix} 14 \\ 11 \\ 10 \end{bmatrix}.
$$

Their solution is $b_0 = 0$, $b_1 = 4$, $b_2 = 3$. The four predictions are $0, 4, 3, 7$, against responses $1, 3, 2, 8$, so

$$
\mathrm{RSS} = 1 + 1 + 1 + 1 = 4.
$$

The additive model uses one slope for $x$ at every $z$. The corners ask for slope $3 - 1 = 2$ in the first row and slope $8 - 2 = 6$ in the second row. One number cannot be both $2$ and $6$, and the residual sum of squares records the compromise.


In [2]:
# Without the product column the residual sum of squares is 4.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
y = np.array([1.0, 3, 2, 8])
X = np.column_stack([np.ones(4), x, z])
beta = np.linalg.solve(X.T @ X, X.T @ y)
predictions = X @ beta
print("additive beta", beta)
print("predictions", predictions)
print("RSS", np.sum((y - predictions) ** 2))
assert np.allclose(beta, [0, 4, 3])
assert np.allclose(predictions, [0, 4, 3, 7])
assert np.sum((y - predictions) ** 2) == 4


additive beta [0. 4. 3.]
predictions [0. 4. 3. 7.]
RSS 4.0


## One change: a shift that is not an interaction

Add $1$ to both responses in the row $z = 1$, so the table becomes $1, 3$ on the first row and $3, 9$ on the second. The slope of $x$ is still $2$ when $z = 0$ and $6$ when $z = 1$. Only the level of the second row changed. Solving again,

$$
b_0 = 1,\quad b_1 = 2,\quad b_2 = 2,\quad b_3 = 4.
$$

The interaction $b_3$ is unchanged. A main-effect shift and a change of slope are different columns, and this edit touched only the main effect of $z$.


In [3]:
# Raising the z=1 row by 1 changes b2 and leaves the interaction at 4.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
y = np.array([1.0, 3, 3, 9])
X = np.column_stack([np.ones(4), x, z, x * z])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta", beta)
assert np.allclose(beta, [1, 2, 2, 4])


beta [1. 2. 2. 4.]


## Pitfall

This four-point interaction model has $n = p = 4$. The residual sum of squares is zero because a full-rank square design can interpolate its responses, just as the cubic did in the polynomial lesson. The slopes $2$ and $6$ are real descriptions of these four numbers. They are not accompanied by a residual variance, because no degree of freedom remains to estimate one. An interaction term is not, by itself, evidence of a stable change in slope beyond the table.


In [4]:
# No residual degrees of freedom remain when p equals n.
n, p = 4, 4
rss = 0
print("residual degrees of freedom", n - p)
print("RSS", rss)
assert n - p == 0


residual degrees of freedom 0
RSS 0


## Summary

- The column $xz$ lets the slope of $x$ be $b_1 + b_3 z$.
- On the four corners, that slope is $2$ when $z = 0$ and $6$ when $z = 1$.
- Without the product, the same corners leave $\mathrm{RSS} = 4$. With $n = p$, a zero residual sum of squares is interpolation.
